# Dark-firm registry evidence review

**Status:** exploratory companion notebook (non-citable)  
**Research question:** B2/B3 — dark-majority follow-up: unresolved commercialization  
outcomes ([docs/research-questions.md](../../docs/research-questions.md))
**Canonical computation:** `scripts/data/nano_ws5b_sam_status.py` (WS5b / T14),  
`scripts/data/nano_ws5c_sector_registries.py` (WS5c / T17)
**Data as of:** the SAM/registry API cache dates recorded by the generating scripts  

Companion view over the registry-evidence artifacts, focused on separating missingness
from negative evidence and on stating each source's coverage before interpreting it.
Exploratory-tier and non-citable.

In [ ]:
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "sbir_etl").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside the sbir-analytics checkout")


REPO_ROOT = find_repo_root()
AREA_ID = "nanotechnology"
REPORT_DIR = REPO_ROOT / "data" / "reports" / AREA_ID
RANDOM_SEED = 20260806

## Data contract

- **Population:** WS5b covers cohort firms *with a known UEI* (plus WS2-resolved
  candidates); WS5c covers only the *biomed slice* of dark firms (any HHS-funded
  award). Neither searched the whole dark population.
- **Grain:** WS5b is entity (UEI) grain; WS5c is firm grain; the liveness file is the
  firm-grain denominator.
- **Keys:** `uei` for SAM status; `firm_normalized` for registries.
- **Missingness rules:** a firm absent from `sam_status.csv` was *not searchable*
  (no UEI), not lapsed. A firm absent from `ws5c_sector_registries.csv` either was not
  in the biomed slice (never searched) or was searched and had no hits — the artifact
  keeps only hit rows, so the denominator must come from the liveness file. An expired
  SAM registration is the closest quasi-negative signal available; registry hits are
  affirmative positive evidence.

In [ ]:
ARTIFACTS = {
    "SAM status (WS5b)": REPORT_DIR / "sam_status.csv",
    "sector registries (WS5c)": REPORT_DIR / "ws5c_sector_registries.csv",
    "dark-firm liveness": REPORT_DIR / "dark_firm_liveness.csv",
}
GENERATORS = {
    "SAM status (WS5b)": "scripts/data/nano_ws5b_sam_status.py (needs SAM_GOV_API_KEY)",
    "sector registries (WS5c)": "scripts/data/nano_ws5c_sector_registries.py",
    "dark-firm liveness": "scripts/data/nano_dark_firm_liveness.py",
}
pd.DataFrame(
    [
        {"artifact": name, "path": str(path.relative_to(REPO_ROOT)), "exists": path.exists()}
        for name, path in ARTIFACTS.items()
    ]
)

def load_artifact(name: str) -> pd.DataFrame:
    """Read a canonical CSV artifact, or return an empty frame with a hint."""
    path = ARTIFACTS[name]
    if not path.exists():
        print(
            f"Missing {path.relative_to(REPO_ROOT)} — artifact not present; "
            f"run {GENERATORS[name]} first."
        )
        return pd.DataFrame()
    return pd.read_csv(path, low_memory=False)

## SAM registration status as dormancy timestamps

Status distribution and expiration-year profile. An expiry year brackets when a firm
stopped seeking federal work; it does not date dissolution.

In [ ]:
sam_status = load_artifact("SAM status (WS5b)")
if sam_status.empty:
    sam_view = pd.DataFrame()
else:
    print("Status distribution:")
    display(sam_status["sam_status"].value_counts(dropna=False).rename("entities").to_frame())
    expiry_years = pd.to_datetime(sam_status["expiration_date"], errors="coerce").dt.year
    sam_view = expiry_years.value_counts().sort_index().rename("registrations_expiring").to_frame()
sam_view

## Sector-registry evidence against its true denominator

WS5c rows are hits only. Reconstruct the searched population (biomed dark firms) from
the liveness file before quoting any rate, and keep clinical-trial and 510(k) channels
separate — they are different market-entry claims.

In [ ]:
registries = load_artifact("sector registries (WS5c)")
liveness = load_artifact("dark-firm liveness")
if registries.empty:
    registry_view = pd.DataFrame()
else:
    registry_view = (
        registries.assign(
            clinical_trial=registries["channels"].str.contains("clinical_trial"),
            fda_510k=registries["channels"].str.contains("fda_510k"),
        )
        .groupby("bucket")[["clinical_trial", "fda_510k"]]
        .sum()
    )
    if not liveness.empty:
        print(
            "Denominator note: hits below are over the searched biomed slice, "
            f"not the full {len(liveness):,}-firm dark population."
        )
registry_view

## Evidence-state ledger

Every firm lands in exactly one state per source: `positive`, `affirmative negative`
(expired SAM), `searched, no hit`, or `not searchable / not searched`. Collapsing the
last two into "negative" is the error this notebook exists to prevent.

In [ ]:
if liveness.empty or sam_status.empty:
    print("Evidence-state ledger needs both the liveness and SAM artifacts present.")
    ledger = pd.DataFrame()
else:
    firm_column = "normalized_name" if "normalized_name" in liveness.columns else "company"
    sam_named = set(sam_status["company"].astype(str).str.strip().str.upper())
    frame = liveness.copy()
    frame["searched_sam"] = frame["company"].astype(str).str.strip().str.upper().isin(sam_named)
    ledger = frame.groupby("searched_sam").size().rename("firms").to_frame()
    print("Name-level join is illustrative only — the canonical key is UEI, and firms")
    print("without one were never searchable in SAM.")
ledger

## Review log

| Firm/sample | Source | Eligible to be searched? | Searched? | State | Follow-up |
|---|---|---|---|---|---|
| _Draft_ | _WS5b or WS5c_ | _UEI / biomed slice_ | _Yes/no_ | _One of the four states_ | _Next instrument_ |